# 04 — Features & Train/Validation Split
Features computed strictly from the feature window; split is group+time aware.

In [1]:
import sys
sys.path.insert(0, "..")
from _lib import load_search_data, build_features, FEATURE_WEEKS
import pandas as pd

df = load_search_data()
features = build_features(df, FEATURE_WEEKS)
labels = pd.read_parquet("data_cache/labels_train.parquet")

data = features.merge(labels, on="page_id", how="inner")
print(data.shape)
data.head()

(500, 17)


,page_id,avg_clicks,avg_impressions,avg_ctr,avg_position,std_clicks,word_count,days_since_publish,trend_slope,cat_blog,cat_comparison,cat_faq,cat_guide,cat_landing,cat_product,label,pct_change
0,page_0000,136.916667,2774.750000,0.049508,7.433333,8.084310,2260,1302,-3.166667,True,False,False,False,False,False,stable,-0.100000
1,page_0001,10.250000,438.166667,0.023358,5.575000,1.864745,1263,161,-1.166667,True,False,False,False,False,False,declining,-0.238095
2,page_0002,70.416667,2986.916667,0.023458,7.508333,17.042505,1147,630,-8.500000,False,False,False,False,True,False,growing,0.605769
3,page_0003,18.500000,473.916667,0.039033,6.858333,2.153222,1493,714,-1.000000,False,False,True,False,False,False,stable,0.055556
4,page_0004,14.250000,285.500000,0.049817,8.500000,2.667140,2510,1190,0.166667,False,False,False,True,False,False,stable,-0.030303


In [2]:
# Time-aware, group-respecting split:
# because features and labels are already built from strictly separate windows
# (weeks 0-11 vs 12-15), a page's full history never spans both sides of the
# leakage boundary. We still split by page_id (group) for train/validation so
# no page appears in both, keeping the split honest for model selection.
from sklearn.model_selection import train_test_split

unique_ids = list(data["page_id"].unique())  # plain python list avoids pyarrow-backed indexing issue
train_ids, val_ids = train_test_split(
    unique_ids, test_size=0.25, random_state=42
)
train = data[data["page_id"].isin(train_ids)]
val = data[data["page_id"].isin(val_ids)]
print("train:", train.shape, "val:", val.shape)

train: (375, 17) val: (125, 17)


In [3]:
train.to_parquet("data_cache/train.parquet")
val.to_parquet("data_cache/val.parquet")
print("Saved train/val splits.")

Saved train/val splits.
